# MyAI — GPU Smoke Run (proves the full pipeline on T4/P100)
1M FineWeb → corpus → 4k BPE → pack-256 → MyAI-20M ×200 steps AMP → eval. ~30-45 min. Full 10M/32k/100M runs after this goes green.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!python scripts/check_env.py

In [ ]:
!python scripts/stream_fineweb.py --max-tokens 1000000 --out data/raw/fineweb-1M.txt
!python scripts/prepare_tokenizer_corpus.py

In [ ]:
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train.txt --vocab-size 4000 --out data/tokenizer_corpus/tokenizer-4k.json

In [ ]:
%%writefile configs/data-pack-smoke.yaml
version: "0.1.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-4k.json
output_dir: data/processed/smoke-4k
seq_len: 256
val_ratio: 0.02


In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-smoke.yaml

In [ ]:
!python scripts/train.py --model-config configs/myai-20m.yaml --data-dir data/processed/smoke-4k --max-steps 200 --batch-size 4 --accum 4 --amp --eval-every 100 --ckpt-every 100 --out-dir /kaggle/working/ckpts-smoke --report /kaggle/working/train-smoke.json

In [ ]:
!python scripts/eval_model.py --checkpoint /kaggle/working/ckpts-smoke/final.pt --data-dir data/processed/smoke-4k --report /kaggle/working/eval-smoke.json
!ls -la /kaggle/working/ckpts-smoke/ && cat /kaggle/working/train-smoke.json